# A forecast without the day's weather

The model reads the weather forecast captured at 08:45 and is asked for its own forecast at 09:00. When that capture is late, the feature table has no weather for the day, and a run still produces a forecast. This notebook asks one question before a fallback is built: **with the day's weather missing, what should the forecast be made from?**

It is the experiment behind the second fault scenario, [F1](../docs/scenarios/F1-late-weather.md). The expected outcome was written there before anything here was run, and is the "We assumed" column of the table at the end.

Four candidates, all scored on the 16 months of the honest result, March 2025 to June 2026, with every day's weather taken away:

| Candidate | What it is |
| --- | --- |
| The model, given empty weather | What a run does today: the same trained model, its five inputs for the day's forecast empty |
| A model trained without the day's weather | The same model fitted on its other four inputs only |
| The seven-day mean with its learned range | The plan's level 2: the baseline `mean_7d_range`, already a model the pipeline can run |
| The seven-day mean alone | The same without a range |

The week's forecast, one of the model's inputs, stays in every candidate: it was read on earlier mornings, and a capture missed today doesn't take it away.

## 0. The model with its weather

**Step.** The rows an honest run is trained on and predicts from, and the model trained month by month as a run trains it: the level every candidate is measured against.

In [1]:
import os
import time
import warnings
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

from jtf.ml.backtest import QUANTILES, by_group, metrics
from jtf.ml.models import ALPHAS, KEPT_INPUTS, columns_of, train_boosted, train_miss_by_hour
from jtf.ml.replay import train_by_month
from jtf.ml.warehouse import BacktestWarehouse

warnings.filterwarnings("ignore", message="BigQuery Storage module not found")
pd.set_option("display.width", 170, "display.max_columns", 30, "display.max_colwidth", 70)
pd.set_option("display.float_format", "{:,.4f}".format)


def setting(name: str) -> str:
    if value := os.environ.get(name):
        return value
    lines = Path("../.env").read_text().splitlines()
    env = dict(line.split("=", 1) for line in lines if "=" in line and not line.startswith("#"))
    return env[name]


PROJECT, PREFIX = setting("JTF_BIGQUERY_PROJECT"), setting("JTF_DBT_DATASET_PREFIX")
warehouse = BacktestWarehouse(PROJECT, setting("JTF_BIGQUERY_LOCATION"), PREFIX)
DWD, DWS, ADS = (f"{PROJECT}.{PREFIX}_{layer}" for layer in ("dwd", "dws", "ads"))
JST = "Asia/Tokyo"


def sql(query: str) -> pd.DataFrame:
    return warehouse.client.query(query).to_dataframe(create_bqstorage_client=False)


FIRST_TEST, LAST = date(2025, 3, 1), date(2026, 6, 30)  # the months the honest 5.21% stands on

# What an honest run is trained on and predicts from, read as a run reads it
known = warehouse.training(LAST, columns_of(KEPT_INPUTS))
days = sql(f"""
    SELECT day AS target_day, sky, is_curtailed AS is_curtailed_day, season
    FROM `{ADS}.ads_ml_day_solar`
""")
known = known.merge(days, on="target_day", how="left", validate="m:1")
known["fold"] = pd.to_datetime(known["target_day"]).dt.to_period("M").dt.start_time.dt.date
in_test = known["fold"] >= FIRST_TEST


def outcomes(said: pd.DataFrame) -> pd.DataFrame:
    """Forecasts for the test rows met by their labels, as ads_ml_outcome_solar gives a run's
    (notebook 11, section 2, sets this beside recorded runs: they agree to 5e-12)."""
    test = known[in_test]
    said = said.reindex(columns=QUANTILES)
    label, capacity = test["label_mw"], test["capacity_mw"]
    error = said["predicted_mw"] - label
    miss = -said.sub(label, axis=0)
    levels = pd.Series(ALPHAS)
    pinball = np.maximum(miss * levels, miss * (levels - 1)).sum(axis=1, skipna=False) / 3
    kept = [
        "fold",
        "decision_time",
        "target_day",
        "slot_start",
        "is_daylight",
        "sky",
        "is_curtailed_day",
        "season",
    ]
    return test[kept].assign(
        error_mw=error,
        error_share=error / capacity,
        in_range=label.between(said["p10_mw"], said["p90_mw"]).where(said["p10_mw"].notna()),
        pinball_share=pinball / capacity,
    )


def read(said: pd.DataFrame) -> dict[str, float]:
    """A forecast's miss overall and by sky, its bias, and how its range does."""
    scored = outcomes(said)
    per_fold, overall = metrics(scored)
    sky = by_group(scored).query("grouping == 'sky'").set_index("group")
    covered = sky["coverage"] if "coverage" in sky else pd.Series(np.nan, index=sky.index)
    return {
        "mae_share": overall["mae_share"],
        "bias_share": overall["bias_share"],
        "clear": sky.loc["clear", "mae_share"],
        "mixed": sky.loc["mixed", "mae_share"],
        "cloudy": sky.loc["cloudy", "mae_share"],
        "coverage": overall.get("coverage", np.nan),
        "coverage, cloudy": covered["cloudy"],
        "pinball_share": overall.get("pinball_share", np.nan),
    }


def by_month(train, rows: pd.DataFrame, asked: pd.DataFrame) -> pd.DataFrame:
    """`train` fitted month by month on `rows`, as a run fits it, and its forecasts for `asked`."""
    return train_by_month(train, rows, asked)[0]


started = time.time()
RESULTS = {
    "level 0: the model with the day's weather": read(
        by_month(train_boosted, known, known[in_test])
    )
}
print(
    f"{len(known):,} labelled slots to {known['target_day'].max()}; "
    f"{int(in_test.sum()):,} in {known.loc[in_test, 'fold'].nunique()} test months"
)
print(
    f"the honest model, trained and scored here in {time.time() - started:.0f} s: "
    f"{RESULTS["level 0: the model with the day's weather"]['mae_share']:.4f} of capacity"
)

41,520 labelled slots to 2026-06-30; 23,376 in 16 test months
the honest model, trained and scored here in 4 s: 0.0521 of capacity


**What it shows.** **Observed:** 5.21% of capacity, the recorded backtest's figure.

## 1. Four ways to forecast without it

**Step.** The five inputs that carry the day's forecast (total, direct and diffuse radiation, cloud cover, temperature) are emptied in every row to be predicted. The rows trained on keep theirs: the history is as it was, only today's capture is late. Each candidate is then trained month by month and scored as a run is.

In [2]:
# The day's weather as a late capture leaves it: empty. The week's forecast stays: it was read on
# earlier mornings
VARIABLES = ("ghi_w_m2", "direct_w_m2", "diffuse_w_m2", "cloud_cover_pct", "temperature_c")
TODAY = [f"forecast_{v}" for v in VARIABLES]
late = known[in_test].assign(**{name: np.nan for name in TODAY})
WITHOUT = tuple(name for name in KEPT_INPUTS if name not in TODAY)
print("inputs of a model trained without the day's weather:", WITHOUT)

RESULTS["the same model, given empty weather"] = read(by_month(train_boosted, known, late))
RESULTS["a model trained without the day's weather"] = read(
    by_month(lambda rows: train_boosted(rows, WITHOUT), known, late)
)
RESULTS["the seven-day mean with its learned range"] = read(
    by_month(train_miss_by_hour, known, late)
)
RESULTS["the seven-day mean alone"] = read(late["solar_mean_7d_mw"].to_frame("predicted_mw"))
pd.DataFrame(RESULTS).T

inputs of a model trained without the day's weather: ('mean_7d_share', 'clear_sky_share', 'forecast_ghi_mean_7d_w_m2', 'half_hour')


,mae_share,bias_share,clear,mixed,cloudy,coverage,"coverage, cloudy",pinball_share
level 0: the model with the day's weather,0.0521,0.0081,0.0322,0.0585,0.0726,0.7311,0.6357,0.0167
"the same model, given empty weather",0.1882,-0.1743,0.2780,0.1868,0.0567,0.1387,0.2928,0.0881
a model trained without the day's weather,0.1014,0.0277,0.0763,0.0616,0.1944,0.7519,0.5279,0.0300
the seven-day mean with its learned range,0.1036,0.0168,0.0834,0.0686,0.1827,0.7874,0.6166,0.0327
the seven-day mean alone,0.1052,0.0018,0.0965,0.0692,0.1683,NaN,NaN,NaN


**What it shows.** **Observed:**

| Candidate | Miss | Bias | Range holds | On cloudy days |
| --- | --- | --- | --- | --- |
| The model with the day's weather | 5.21% | +0.8 | 73% | 64% |
| The model, given empty weather | **18.82%** | −17.4 | **14%** | 29% |
| A model trained without the day's weather | 10.14% | +2.8 | 75% | 53% |
| The seven-day mean with its learned range | 10.36% | +1.7 | 79% | 62% |
| The seven-day mean alone | 10.52% | +0.2 | – | – |

- **Given empty weather, the model is far worse than any fallback:** it misses by 18.8% of capacity, 17 points too low, and its range holds one outcome in seven
- **The two fallbacks that know nothing of the day are close:** 10.1% and 10.4%. Both are twice the model's error
- **The seven-day mean's range is the more honest:** it holds 79%, and 62% on cloudy days where the other holds 53%

**Learned → design:**

- **Level 2 is the seven-day mean with its learned range,** as the plan has it. A model trained without the day's weather is 0.2 points better and would be a second model to train, keep and explain, with a range that is worse where it matters
- **What a run does today is the worst of the four.** The next step looks at why

## 2. What the model says when its weather is empty

**Step.** The model's forecast with and without the day's weather, as shares of capacity: its mean at four hours of the day, and how much it varies from day to day at noon.

In [3]:
# What the model says when its weather is empty, beside what it says with it, by half hour
with_weather = by_month(train_boosted, known, known[in_test])
without = by_month(train_boosted, known, late)
test = known[in_test]
day = test["is_daylight"].astype(bool)
capacity = test.loc[day, "capacity_mw"]


def share(mw: pd.Series) -> pd.Series:
    return mw[day] / capacity


shape = pd.DataFrame(
    {
        "label": share(test["label_mw"]),
        "with weather": share(with_weather["predicted_mw"]),
        "empty weather": share(without["predicted_mw"]),
        "seven-day mean": share(test["solar_mean_7d_mw"]),
    }
)
hour = test.loc[day, "slot_start"].dt.tz_convert(JST).dt.hour
print("spread across days at 12:00 to 13:00, as shares of capacity (standard deviation):")
print(shape[hour == 12].std().round(3).to_dict())
shape.groupby(hour).mean().loc[[6, 9, 12, 15]].round(3)

spread across days at 12:00 to 13:00, as shares of capacity (standard deviation):
{'label': 0.22, 'with weather': 0.193, 'empty weather': 0.023, 'seven-day mean': 0.111}


,label,with weather,empty weather,seven-day mean
slot_start,,,,
6,0.0590,0.0620,0.0360,0.0580
9,0.4100,0.4190,0.1430,0.4130
12,0.5090,0.5200,0.1610,0.5130
15,0.2250,0.2380,0.1070,0.2270


**What it shows.** **Observed:**

- **At noon the model with empty weather says 16% of capacity on average,** where 51% came and the seven-day mean says 51%
- **It says nearly the same every day:** its noon forecast varies by 0.02 of capacity from day to day, against 0.19 with weather and 0.22 in what happened

**Learned → design:** LightGBM sends a missing value down the side of each split that fitted its training rows better. No training row lacks the day's weather in daylight, so for radiation that side is no learned answer, and here it is the dark one: the model reads an absent forecast as a forecast of almost no sun. **A missing forecast must never reach the model.** The run has to see that the weather is missing and answer from level 2, and say that it did.

## What this exploration made known

| We assumed | We found | So |
| --- | --- | --- |
| Level 2, the seven-day mean with its learned range, misses by about 10.4% with a range that holds about 79% | 10.36% and 78.7%: its own recorded backtest (section 1) | **Level 2 is `mean_7d_range`.** A late capture doubles the error and keeps the range honest |
| The model given empty weather is worse than level 2 and badly calibrated | Far worse: 18.82%, 17 points too low, its range holding 14%. It reads a missing forecast as a dark day (sections 1, 2) | **A run must not hand the model empty weather.** Today it does, and nothing says so |
| A model trained without the day's weather does about 10.1%, a little better than level 2 | 10.14%, with a range that holds 75%, and 53% on cloudy days (section 1) | Not worth a second model |

## Still open

| What | Why it can't be known yet | When |
| --- | --- | --- |
| What level 1 would cost: an older forecast that still covers the day | The morning capture reaches to 07:30 two days ahead, so yesterday's stops before the target day's daylight | After the capture asks for one more day, and some weeks of captures |
| How often the capture is late | Six mornings captured so far, none late | With months of capture |

## Sources

| Page | Supports |
| --- | --- |
| [F1's write-up](../docs/scenarios/F1-late-weather.md) | The expected outcome, written before this was run |
| [Notebook 11](11_inputs_of_the_solar_model.ipynb), sections 2 and 4.2 | The scoring used here; the model without the day's forecast |
| `jtf/src/jtf/ml/models.py` | The model and the seven-day mean with its learned range |